# Push-up form classifier - Colab training runner

Same flow as squat, BUT the Kaggle dataset `mohamadashrafsalama/pushup` schema is **UNVERIFIED**.
So run the **INSPECT** cell first and read its output: it prints the real files + columns + label
values. Only if they match our 10 rep-level features + a `correct/incorrect` label will the TRAIN
cell succeed as-is; otherwise paste the INSPECT output back and we'll align
`ml/src/healthtrainer_ml/pushup_pose_dataset.py` (column names / aggregation) before training.

Run top to bottom. Mount Drive with the **kimgt2828** account; complete the Kaggle login form.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
print(DRIVE_ROOT)

In [ ]:
# Clone the model-training branch (has the push-up code). %cd /content first so re-running is safe.
%cd /content
!rm -rf /content/health_trainer
!git clone --branch model-training --single-branch https://github.com/kimgt0128/health-trainer.git /content/health_trainer
%cd /content/health_trainer
!pip install -q "kagglehub[pandas-datasets]"

In [ ]:
# Kaggle auth - enter username + API token, wait for the green confirmation, then continue.
import kagglehub
kagglehub.login()

In [ ]:
# === INSPECT (run FIRST) - reveal the real dataset schema, since it's unverified locally. ===
import os, pandas as pd, kagglehub

ds = kagglehub.dataset_download("mohamadashrafsalama/pushup")
print("dataset root:", ds, "\n--- files ---")
all_files = []
for root, _, files in os.walk(ds):
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), ds)
        all_files.append(rel)
        print("  ", rel)

csvs = [os.path.join(ds, f) for f in all_files if f.lower().endswith(".csv")]
for c in csvs[:3]:
    print("\n===== CSV:", os.path.relpath(c, ds), "=====")
    df = pd.read_csv(c)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print(df.head(3).to_string())
    # candidate label columns (object dtype or few unique values)
    for col in df.columns:
        n = df[col].nunique(dropna=True)
        if df[col].dtype == object or n <= 12:
            print(f"  label? {col} ({n} uniq): {list(pd.unique(df[col]))[:12]}")

In [ ]:
# === TRAIN (run after INSPECT confirms / after we align the adapter). ===
# Succeeds only if pushup_pose_dataset.FEATURE_COLUMNS + 'label' match the real CSV. If it raises
# 'missing columns', the INSPECT output above tells us what to map -> fix pushup_pose_dataset.py.
%cd /content/health_trainer
import os
os.environ['PYTHONPATH'] = '/content/health_trainer/ml/src'
!python ml/src/train_pushup_form_classifier.py \
    --run-dir "$RUNS_DIR/pushup_form_classifier_v1" \
    --test-size 0.2 \
    --random-state 42

In [ ]:
# Inspect artifacts written to Drive (4 files expected).
!find "$RUNS_DIR/pushup_form_classifier_v1" -maxdepth 1 -type f -print
!cat "$RUNS_DIR/pushup_form_classifier_v1/metrics_summary.json"